# 🧪 37 · Protein sequences: from coevolution to protein language models
### Conservation and co-variation in a protein family, mutual information and APC, Potts models and direct-coupling analysis (mean-field and pseudo-likelihood), zero-shot variant effects and epistasis, a family-specific language model, and the non-linear readout of deep mutational scans

**BiologyToML** · Level 2 · Research frontier · Part H — Genomes, proteins and molecules · Module 37 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/37_protein_sequences_language_models.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, PyTorch (CPU), scikit-learn, pandas, Matplotlib |
| **Prerequisites** | Level 1 (especially Modules 05, 18 and 20) and [Module 24](24_attention_transformers.ipynb) (attention, transformers, masked language models) |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [05 · Likelihood](05_likelihood_mle_map.ipynb), [15 · Bayesian networks](15_bayesian_networks_representation.ipynb), [17 · Inference and learning](17_bayesian_networks_inference_learning.ipynb), [18 · Gaussian graphical models](18_gaussian_graphical_models.ipynb), [20 · HMM learning (profile HMMs)](20_hmm_learning.ipynb), [21 · Bayesian learning and MCMC](21_bayesian_learning_mcmc.ipynb), [24 · Attention and transformers](24_attention_transformers.ipynb) |
| **Research frontier covered** | multiple sequence alignments, coevolution and direct-coupling analysis, Potts models, contact prediction, zero-shot variant-effect prediction, deep mutational scanning, protein language models — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All data are **SIMULATED** with the course toolkit: `simulate_helix_bundle` (a 68-residue four-helix bundle and its
> true contact map), `simulate_protein_family` (3,000 aligned sequences drawn from a Potts model whose couplings sit
> exactly on the contacts; the true fields $h$ and couplings $J$ are known) and `deep_mutational_scan` (every single
> mutant of the wild type, with its true effect $\Delta E$ and a noisy, sigmoidal "measured" fitness). The simulated
> alignment has no gaps, no insertions and no phylogeny; real alignments have all three (Sections 1 and 8 explain what
> changes).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a multiple sequence alignment records (conservation and co-variation) and why real alignments are reweighted; **compute** sequence weights and the effective number of sequences.
2. **Build** an independent-site model (a PSSM, the profile of Module 20) with pseudocounts and **use** it to score mutations zero-shot, validated against a deep mutational scan and the true effects.
3. **Compute** mutual information between alignment columns, **apply** the average product correction and **evaluate** contact predictions by precision@L/2 against the true contact map.
4. **Explain** why mutual information is fooled by indirect chains $i$–$j$–$k$, and **show** with an exact calculation that the inverse covariance of one-hot columns — the categorical cousin of Module 18's precision matrix — is not.
5. **Write** the Potts model, **explain** why its likelihood is intractable, and **infer** couplings by mean-field DCA and pseudo-likelihood DCA (in PyTorch); **validate** them against the true couplings and alignment depth.
6. **Predict** single- and double-mutant effects with the fitted Potts model (the EVmutation idea) and **quantify** the epistasis it captures that additive models miss.
7. **Train** a family-specific masked language model, **compare** it with the Potts model, and **explain** from the structure of the couplings when a small transformer can or cannot learn them.
8. **Explain** global epistasis (the non-linear readout of DMS assays), why benchmarks use Spearman correlation, and **fit** a monotone non-linearity on top of a predictor.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · 🔬 Families and alignments | simulate a family; conservation; sequence reweighting and N_eff | 20 min |
| 2 · Independent-site model | PSSM with pseudocounts; 🧪 zero-shot scores vs DMS and true ΔE | 15 min |
| 3 · Covariation and contacts | mutual information, APC, precision@L/2; indirect chains (exact) | 25 min |
| 4 · Potts models and DCA | mean-field DCA, pseudo-likelihood DCA; 🧪 contacts, couplings, depth | 35 min |
| 5 · Zero-shot variant effects | Potts ΔE vs independent vs truth; double mutants and epistasis | 20 min |
| 6 · A protein language model | masked LM on the family; masked marginals vs Potts; why it fails here | 25 min |
| 7 · Global epistasis | sigmoid readout, Spearman vs Pearson, fitting a monotone link | 15 min |
| 8 · 🧭 From toy to real data | Pfam, UniRef, EVcouplings, ESM, EVE, AlphaMissense, ProteinGym, ACMG | 10 min |
| ✅ / 🧠 / 🚀 / 📝 | takeaways, checkpoint, discussion, going further, lab note | 10 min |

## 🧫 The biological question

A clinical geneticist sequences a child with a suspected inherited disorder and finds a missense variant — one amino
acid changed — in a gene known to cause the disease. The variant has never been seen before. Is it damaging? Many
missense variants in clinical databases end up classified as *variants of uncertain significance*, because no
experiment has tested them.

Evolution, however, has been testing variants for a billion years. Align the sequences of the same protein from
thousands of species and two patterns appear. Some positions hardly ever change: they are **conserved**, and mutations
there are likely to hurt. And some *pairs* of positions change together: when one residue changes, its partner changes
in a compensating way, because the two residues **touch** in the folded protein. In 2011 several groups showed that
these co-varying pairs, once direct couplings are separated from indirect correlations by **direct-coupling analysis
(DCA)**, predict contacts accurately enough to fold proteins (Morcos *et al.* 2011; Marks *et al.* 2011). In 2017 the
same statistical models were shown to predict mutation effects measured by **deep mutational scans** (EVmutation, Hopf
*et al.* 2017), and the line of work continued through deep generative models (DeepSequence, EVE) to protein language
models (ESM-1v) and AlphaMissense, which now provide pathogenicity predictions for essentially every possible human
missense variant (Cheng *et al.* 2023).

This notebook walks that path on a SIMULATED protein family whose true structure, couplings and mutation effects are
known — a miniature of a Pfam alignment plus a ProteinGym-style deep mutational scan:

1. conservation → an independent-site model; 2. co-variation → mutual information; 3. direct couplings → Potts models
and DCA; 4. zero-shot variant effects and epistasis; 5. a protein language model; 6. the non-linear readout of the assay.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import curve_fit
from scipy.special import logsumexp as lse
from scipy.stats import pearsonr, spearmanr
from sklearn.isotonic import IsotonicRegression

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration and colours ──
torch.manual_seed(SEED)
torch.set_num_threads(2)                 # small models: 2 CPU threads are plenty
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#E69F00", "#CC79A7", "#56B4E9"]   # colour-blind-safe palette
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads())

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_helix_bundle` gives the structure and contact map, `simulate_protein_family` samples an alignment from a Potts model on that contact map, `potts_score` evaluates the Potts model, and `deep_mutational_scan` simulates a measurement of every single mutant. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"


def simulate_helix_bundle(n_helices: int = 4, helix_length: int = 14, loop_length: int = 4,
                          rng=None, noise: float = 0.0) -> dict:
    """Cα coordinates (Å) of an idealised four-helix-bundle protein, its contact map and secondary structure.

    Helices are ideal right-handed α-helices (3.6 residues per turn, 1.5 Å rise, 2.3 Å radius)
    packed antiparallel on the corners of a 10 Å square and joined by short loops.  ``noise`` adds
    Gaussian jitter (needs ``rng``).  Contacts: Cα–Cα distance < 8 Å and |i − j| ≥ 3.

    Returns a dict with ``coords`` (L × 3), ``ss`` (string of 'H'/'L'), ``distances`` and ``contacts``.
    """
    corners = 5.0 * np.array([[1, 1], [1, -1], [-1, -1], [-1, 1]], float)
    pts, ss = [], ""
    for h in range(n_helices):
        cx, cy = corners[h % 4] * (1 + h // 4)
        up = h % 2 == 0
        k = np.arange(helix_length)
        angle = np.deg2rad(100.0) * k * (1 if up else -1)      # turn with the direction of travel, so
        z = 1.5 * (k - (helix_length - 1) / 2) * (1 if up else -1)   # every helix is right-handed
        helix = np.c_[cx + 2.3 * np.cos(angle), cy + 2.3 * np.sin(angle), z]
        if pts:                                                      # loop: circular arc, 3.8 Å steps
            start, end = pts[-1], helix[0]
            d, m = np.linalg.norm(end - start), loop_length + 1
            lo, hi = 1e-6, 2 * np.pi - 1e-6                              # arc angle Θ with m equal chords
            for _ in range(60):
                theta = (lo + hi) / 2
                lo, hi = (theta, hi) if 3.8 * np.sin(theta / 2) / np.sin(theta / (2 * m)) > d else (lo, theta)
            R = 1.9 / np.sin(theta / (2 * m))
            e1 = (end - start) / d
            e2 = np.r_[0, 0, np.sign(start[2])] - np.dot(np.r_[0, 0, np.sign(start[2])], e1) * e1
            e2 /= np.linalg.norm(e2)
            c = (start + end) / 2 - R * np.cos(theta / 2) * e2
            for k in range(1, m):
                psi = -theta / 2 + k * theta / m
                pts.append(c + R * (np.sin(psi) * e1 + np.cos(psi) * e2))
            ss += "L" * loop_length
        pts.extend(helix)
        ss += "H" * helix_length
    coords = np.array(pts)
    if noise > 0:
        coords = coords + rng.normal(0, noise, coords.shape)
    D = np.linalg.norm(coords[:, None] - coords[None], axis=-1)
    sep = np.abs(np.arange(len(coords))[:, None] - np.arange(len(coords))[None])
    return dict(coords=coords, ss=ss, distances=D, contacts=(D < 8.0) & (sep >= 3))


def potts_score(seqs, h, J) -> np.ndarray:
    """Statistical fitness Σ_i h_i(a_i) + Σ_{i<j} J_ij(a_i, a_j) of integer-coded sequences (higher = fitter)."""
    seqs = np.atleast_2d(seqs)
    L = seqs.shape[1]
    ii, jj = np.nonzero(np.triu(np.abs(J).sum(axis=(2, 3)) > 0, 1))      # coupled pairs only
    return h[np.arange(L), seqs].sum(axis=1) + J[ii, jj, seqs[:, ii], seqs[:, jj]].sum(axis=1)


def simulate_protein_family(rng: np.random.Generator, contacts: np.ndarray, n_seqs: int = 3000,
                            q: int = 20, field_scale: float = 1.0, coupling_scale: float = 1.0,
                            n_sweeps: int = 100) -> dict:
    """Simulate a protein family (a multiple sequence alignment) from a Potts model on a contact map.

    Each residue pair in contact gets a q × q coupling matrix J_ij (pairs that fit together
    are rewarded), each position a field h_i (its conservation).  Sequences are drawn by
    Gibbs sampling (``n_seqs`` independent chains, ``n_sweeps`` sweeps each), so conserved
    positions and *co-varying* contact pairs appear exactly as in natural alignments — but
    without phylogeny.  The wild type is the fittest sampled sequence.

    Returns a dict with ``msa`` (n_seqs × L integers 0…q−1; letters = AMINO_ACIDS[:q]),
    ``h`` (L × q), ``J`` (L × L × q × q, symmetric: J[j, i] = J[i, j].T), ``wild_type``.
    """
    L = contacts.shape[0]
    cons = rng.uniform(0.3, 1.5, L)[:, None] * field_scale
    h = rng.normal(0, 1, (L, q)) * cons
    J = np.zeros((L, L, q, q))
    pairs = [(i, j) for i in range(L) for j in range(i + 1, L) if contacts[i, j]]
    for i, j in pairs:
        J[i, j] = rng.normal(0, 0.5 * coupling_scale, (q, q))
        J[j, i] = J[i, j].T
    nbrs = [np.flatnonzero(contacts[i]) for i in range(L)]
    x = rng.integers(0, q, (n_seqs, L))
    rows = np.arange(n_seqs)
    for _ in range(n_sweeps):
        for i in rng.permutation(L):
            logits = np.tile(h[i], (n_seqs, 1))
            for j in nbrs[i]:
                logits += J[j, i][x[:, j]]                                 # = J_ij(·, x_j)
            g = logits - np.log(-np.log(rng.random((n_seqs, q))))           # Gumbel-max sampling
            x[rows, i] = np.argmax(g, axis=1)
    wild_type = x[np.argmax(potts_score(x, h, J))].copy()
    return dict(msa=x, h=h, J=J, wild_type=wild_type)


def deep_mutational_scan(rng: np.random.Generator, wild_type, h, J, noise: float = 0.15) -> pd.DataFrame:
    """Simulate a deep mutational scan: every single mutant of ``wild_type`` with a measured fitness.

    True effect ΔE = score(mutant) − score(wild type) under the Potts model; the assay reads it
    through a sigmoid ("fraction folded", global epistasis) plus measurement noise.
    Returns a DataFrame with position, wild-type and mutant letters, ``delta_energy`` and ``fitness``.
    """
    wt = np.asarray(wild_type)
    L, q = h.shape
    # ΔE for every substitution i: wt_i → a, computed in one go (only position i changes)
    pair = J[np.arange(L)[:, None], np.arange(L)[None, :], :, wt[None, :]].sum(axis=1)    # L × q
    delta = (h - h[np.arange(L), wt][:, None]) + (pair - pair[np.arange(L), wt][:, None])
    rows = [(i, AMINO_ACIDS[wt[i]], AMINO_ACIDS[a], delta[i, a]) for i in range(L) for a in range(q) if a != wt[i]]
    df = pd.DataFrame(rows, columns=["position", "wt", "mut", "delta_energy"])
    df["fitness"] = 1 / (1 + np.exp(-(df["delta_energy"] + 3.0) / 1.5)) + rng.normal(0, noise, len(df))
    df["mutant"] = df["wt"] + df["position"].astype(str) + df["mut"]
    return df


print("🧰 Toolkit loaded:", ", ".join(['simulate_helix_bundle', 'potts_score', 'simulate_protein_family', 'deep_mutational_scan']))

---
## 1 · 🔬 Families and alignments

> 📍 **Where we are.** [Module 24](24_attention_transformers.ipynb) introduced protein families as training data for
> language models. Here the family itself is the object of study: what can be read from the columns of an alignment,
> one at a time and in pairs?

🔬 **Assay box — from sequences to an alignment.** *What is measured:* genomes and metagenomes are sequenced; proteins
are predicted from them and deposited in UniProt. *How a family is built:* a query sequence is searched against the
database with profile methods (jackhmmer from HMMER, HHblits), hits are aligned column by column, and columns with
too many gaps are dropped. Curated family alignments are collected in Pfam. *Typical sizes:* $10^2$–$10^5$ sequences
for well-studied families. *Typical artefacts:* gaps from insertions and deletions, misaligned regions, fragments, and
**phylogenetic redundancy** — dozens of near-identical sequences from closely related strains, which make some
sequences count far more than they should.

The toolkit simulates such a family: the 68-residue four-helix bundle of `simulate_helix_bundle` provides the contact
map, and `simulate_protein_family` samples 3,000 sequences from a Potts model in which every pair of residues in contact
has its own coupling table (Section 4 explains the model). We fit every family model on 2,000 sequences and keep 1,000
aside for held-out checks.

In [ ]:
# ── 1.1 Structure and family (SIMULATED; ≈ 7 s) ──
structure = simulate_helix_bundle()
contacts, ss = structure["contacts"], structure["ss"]
L, Q = contacts.shape[0], 20
sep = np.abs(np.arange(L)[:, None] - np.arange(L)[None, :])
long_range = contacts & (sep >= 6)                        # contacts that are not helix neighbours (i ± 3, i ± 4)
rng = np.random.default_rng(SEED)
t0 = time.time()
family = simulate_protein_family(rng, contacts, n_seqs=3000)
msa, h_true, J_true, wild_type = family["msa"], family["h"], family["J"], family["wild_type"]
msa_fit, msa_held = msa[:2000], msa[2000:]                 # fit set and held-out set
print(f"SIMULATED alignment: {msa.shape[0]} sequences × {L} positions in {time.time() - t0:.1f} s")
for k in range(3):
    print("  ", "".join(AMINO_ACIDS[a] for a in msa[k]))
print("wild type (fittest sampled sequence):", "".join(AMINO_ACIDS[a] for a in wild_type))
print(f"contacts: {contacts.sum() // 2} pairs ({long_range.sum() // 2} with |i − j| ≥ 6); "
      f"coupled pairs in the true model: {(np.abs(J_true).sum(axis=(2, 3)) > 0).sum() // 2}")

In [ ]:
# ── 1.2 Conservation: column entropy vs the true strength of each position's preferences ──
def column_frequencies(msa, pseudocount=0.0, weights=None):
    """f_i(a): frequency of residue a in column i (optionally weighted), mixed with a uniform pseudocount."""
    w = np.ones(len(msa)) if weights is None else weights
    f = np.stack([np.bincount(msa[:, i], weights=w, minlength=Q) for i in range(msa.shape[1])]) / w.sum()
    return (1 - pseudocount) * f + pseudocount / Q


f_fit = column_frequencies(msa_fit)
entropy = -(f_fit * np.log2(np.where(f_fit > 0, f_fit, 1))).sum(1)       # bits; log2(20) = 4.32 = uniform
field_spread = h_true.std(1)                                              # truth: how picky position i is
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [2, 1]})
a1.bar(range(L), entropy, color=[OKABE_ITO[0] if s == "H" else OKABE_ITO[3] for s in ss])
a1.axhline(np.log2(20), color="k", ls="--", lw=0.8)
a1.set_xlabel("alignment column (position)"); a1.set_ylabel("entropy (bits)")
a1.set_title("SIMULATED family: conservation per column (blue = helix, orange = loop)")
a2.scatter(field_spread, entropy, s=14, color=OKABE_ITO[0])
a2.set_xlabel("true field spread  sd$_a\\,h_i(a)$"); a2.set_ylabel("entropy (bits)")
a2.set_title(f"ground truth: Spearman ρ = {spearmanr(field_spread, entropy)[0]:.2f}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Column entropy varies from position to position, and it tracks the truth: positions whose
true fields $h_i(a)$ are spread out (strong preferences) have low entropy. The correlation is strong but not perfect,
because the couplings also shape each column. Note that in this simulation conservation is not tied to secondary
structure (helix and loop columns look alike); in real proteins, buried core positions and active sites are the most
conserved.

**Sequence reweighting.** The simulated sequences are independent draws, but real alignments are not: a family may
contain 300 sequences from closely related *E. coli* strains and one from an archaeon. Counting every sequence once
lets the over-sampled clade dominate the frequencies. The standard fix (used by DCA and EVmutation) gives each sequence
a weight $w_n=1/m_n$, where $m_n$ is the number of sequences at least 80 % identical to sequence $n$ (itself included),
and reports the **effective number of sequences** $N_{\text{eff}}=\sum_n w_n$.

In [ ]:
# ── 1.3 Reweighting: one sequence over-sampled 300 times ──
def sequence_weights(msa, threshold=0.8):
    """w_n = 1 / (number of sequences with ≥ threshold identity to sequence n, itself included)."""
    X = np.eye(Q)[msa].reshape(len(msa), -1)
    identity = X @ X.T / msa.shape[1]                       # fraction of identical positions for every pair
    return 1.0 / (identity >= threshold).sum(1)


clean = msa_fit[:1000]
biased = np.vstack([clean, np.repeat(msa_fit[1000:1001], 300, axis=0)])   # a clade sequenced 300 extra times
w = sequence_weights(biased)
reference = column_frequencies(msa_fit)                      # frequencies of the unbiased family (2,000 sequences)
err = lambda f: np.abs(f - reference).sum(1).mean() / 2     # average total-variation distance per column
print(f"sequences: {len(biased)}   effective number N_eff = Σ w = {w.sum():.1f}")
print(f"weights of the 300 copies: {w[-1]:.4f} each;  clean sequences with weight < 1: {(w[:1000] < 1).sum()}")
print(f"column-frequency error vs the unbiased family:  clean 1,000 {err(column_frequencies(clean)):.3f} | "
      f"biased, unweighted {err(column_frequencies(biased)):.3f} | biased, reweighted {err(column_frequencies(biased, weights=w)):.3f}")

🔍 **What just happened?** Adding 300 copies of one sequence distorted the column frequencies badly (the average error
grows almost six-fold, from 0.035 to 0.20). The 301 identical sequences each received weight $1/301$, so together they count as one sequence, and
$N_{\text{eff}}$ drops back to about 1,001. The reweighted frequencies are as good as those of the clean alignment. No
clean sequence was down-weighted: two independent members of our family share far less than 80 % of their residues.

⚠️ **Pitfall.** Reweighting removes *redundancy*, not *phylogeny*. Sequences from one clade that are 60 % identical
still share history, and pairs of positions can co-vary simply because they mutated together on the same branch of the
tree — a source of false couplings that our phylogeny-free simulation cannot show (🚀 Bonus).

✏️ **Your turn.** Repeat cell 1.3 with `threshold=0.6` and `threshold=0.95`. How do $N_{\text{eff}}$ and the error
change? What would happen to the copies if they each carried two or three random mutations?

---
## 2 · The independent-site model

> 📍 **Where we are.** The simplest model of a family treats every column separately: residue $a$ at position $i$ has
> probability $f_i(a)$, independent of all other positions. This is the **profile** behind PSSMs and the match-state
> emissions of the profile HMMs of [Module 20](20_hmm_learning.ipynb), Section 7.

**The model and its scores.** With pseudocounts (the MAP estimate with a Dirichlet prior of
[Module 08](08_knn_naive_bayes.ipynb), Section 8) the column probabilities are
$\hat f_i(a)=(1-\lambda)\,n_i(a)/N+\lambda/20$. A **position-specific scoring matrix** compares them with a background
frequency $b(a)$, and an independent-site **mutation score** compares the mutant with the wild-type residue:

$$\mathrm{PSSM}_i(a)=\log\frac{\hat f_i(a)}{b(a)},\qquad
s^{\text{ind}}(i,\,a_{\text{wt}}\!\to a)=\log\hat f_i(a)-\log\hat f_i(a_{\text{wt}}).$$

*In words:* the PSSM says how much more often residue $a$ appears at position $i$ than in proteins in general — used
to *find* family members; the mutation score says how much rarer the mutant residue is than the wild-type residue in
this column of the family — used to *score variants*. A strongly negative score means "evolution rarely tolerates this
substitution here".

> 🔗 **Builds on** [Module 20 · HMM learning](20_hmm_learning.ipynb), Section 7: the match states of a profile HMM emit
> residues with exactly these position-specific probabilities; a PSSM is a profile HMM without insertions and deletions.

In [ ]:
# ── 2.1 PSSM of the family and the true effect of every substitution ──
LAMBDA = 0.02                                                  # pseudocount weight (MAP smoothing)
f_hat = column_frequencies(msa_fit, pseudocount=LAMBDA)
background = np.bincount(msa_fit.ravel(), minlength=Q) / msa_fit.size
pssm = np.log(f_hat / background)                              # L × 20 log-odds
rng = np.random.default_rng(SEED + 2)
dms = deep_mutational_scan(rng, wild_type, h_true, J_true)     # SIMULATED DMS: every single mutant of the wild type
true_map = np.full((L, Q), np.nan)
true_map[dms.position.to_numpy(), dms.mut.map(AMINO_ACIDS.index).to_numpy()] = dms.delta_energy.to_numpy()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.4))
for ax, M, title, cmap in [(a1, pssm.T, "PSSM log-odds (family columns)", "RdBu_r"),
                           (a2, true_map.T, "true ΔE of every substitution of the wild type (SIMULATED)", "RdBu_r")]:
    v = np.nanmax(np.abs(M))
    im = ax.imshow(M, aspect="auto", cmap=cmap, vmin=-v, vmax=v); fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_yticks(range(Q)); ax.set_yticklabels(list(AMINO_ACIDS), fontsize=6)
    ax.set_xlabel("position"); ax.set_title(title)
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.2 🧪 Zero-shot scores of the independent-site model vs the DMS and the truth ──
pos = dms["position"].to_numpy()
mut = dms["mut"].map(AMINO_ACIDS.index).to_numpy()
wt_aa = wild_type[pos]
log_f = np.log(f_hat)
dms["independent"] = log_f[pos, mut] - log_f[pos, wt_aa]
rho = lambda a, b: spearmanr(a, b)[0]
print(f"{len(dms)} single mutants of the wild type")
print(f"Spearman ρ(independent-site score, measured fitness) = {rho(dms.independent, dms.fitness):.3f}")
print(f"Spearman ρ(independent-site score, true ΔE)          = {rho(dms.independent, dms.delta_energy):.3f}")
print(f"ceiling: Spearman ρ(true ΔE, measured fitness)        = {rho(dms.delta_energy, dms.fitness):.3f}")

🔍 **What just happened?** The PSSM (left) and the true mutational landscape (right) share their broad pattern:
positions where the family is picky are positions where most substitutions hurt. Quantitatively, the independent-site
score orders the 1,292 mutants with Spearman $\rho\approx0.58$ against the measured fitness and $\approx0.67$ against the
true effects (exact values above). The **ceiling** line is important: even the true $\Delta E$ reaches only
$\rho\approx0.87$ with the measured fitness, because the simulated assay is noisy and saturating (Section 7). No predictor
should be expected to beat that.

What does the independent model miss? The true effect of a substitution at $i$ is
$\Delta E=h_i(a)-h_i(a_{\text{wt}})+\sum_j\big[J_{ij}(a,x^{\text{wt}}_j)-J_{ij}(a_{\text{wt}},x^{\text{wt}}_j)\big]$: the field
part is what conservation sees, the coupling part depends on the residues at the contact partners *in this wild type*.
To capture it we must model pairs.

✏️ **Your turn.** Set `LAMBDA = 0.5` and recompute. Large pseudocounts flatten the profile; does the correlation get
better or worse? Then try fitting the profile on only 50 sequences (`msa_fit[:50]`) with `LAMBDA = 0.02` and with
`LAMBDA = 0.3`: when do pseudocounts help?

---
## 3 · Covariation and contacts

> 📍 **Where we are.** Pairs of columns that co-vary should be pairs of residues in contact. The simplest measure of
> co-variation is the **mutual information** (MI) between two columns — the quantity behind many early coevolution
> contact predictors.

$$\mathrm{MI}_{ij}=\sum_{a,b}f_{ij}(a,b)\,\log\frac{f_{ij}(a,b)}{f_i(a)\,f_j(b)}.$$

*In words:* MI compares the joint frequencies of residue pairs in columns $i$ and $j$ with what independence would
predict, $f_i(a)f_j(b)$. It is zero if and only if the columns are independent, and it grows the more knowing one column
tells you about the other. (It is the Kullback–Leibler divergence between the joint distribution and the product of the
marginals.)

**Scoring contacts.** As in Module 24 we rank all pairs with $|i-j|\ge6$ and report the **precision of the top
$L/2=34$**: the fraction that are true contacts. We also apply the **average product correction**
$\mathrm{APC}(S)_{ij}=S_{ij}-\bar S_{i\cdot}\bar S_{\cdot j}/\bar S_{\cdot\cdot}$ (Dunn, Wahl & Gloor 2008), which removes the
part of a pair's score explained by how high each column scores with *everything*.

In [ ]:
# ── 3.1 Pair frequencies and mutual information ──
def pair_frequencies(msa, pseudocount):
    """f_i(a) (L × 20) and f_ij(a, b) (L × L × 20 × 20) with a uniform pseudocount; f_ii = diag(f_i)."""
    N = len(msa)
    X = np.eye(Q)[msa].reshape(N, -1)                                  # one-hot, N × (L·20)
    fij = (X.T @ X / N).reshape(L, Q, L, Q).transpose(0, 2, 1, 3)
    fi = (1 - pseudocount) * X.mean(0).reshape(L, Q) + pseudocount / Q
    fij = (1 - pseudocount) * fij + pseudocount / Q ** 2
    for i in range(L):
        fij[i, i] = np.diag(fi[i])
    return fi, fij


def mutual_information(msa, pseudocount=0.02):
    """MI between every pair of columns (diagonal set to 0)."""
    fi, fij = pair_frequencies(msa, pseudocount)
    indep = fi[:, None, :, None] * fi[None, :, None, :]
    for i in range(L):
        fij[i, i] = indep[i, i]                                         # MI(i, i) is not needed: make it 0
    MI = (fij * np.log(fij / indep)).sum(axis=(2, 3))
    np.fill_diagonal(MI, 0)
    return MI

In [ ]:
# ── 3.2 Contact-scoring helpers, and how well MI ranks contacts ──
def apc_correct(S):
    """Average product correction: S_ij − mean_i · mean_j / overall mean (diagonal ignored)."""
    S = S.copy(); np.fill_diagonal(S, 0)
    row = S.sum(1) / (L - 1)
    return S - np.outer(row, row) / (S.sum() / (L * (L - 1)))


def precision_at_k(scores, k=L // 2, min_sep=6):
    """Fraction of true contacts among the k top-scoring pairs with |i − j| ≥ min_sep."""
    iu = np.triu_indices(L, min_sep)
    return contacts[iu][np.argsort(-scores[iu])[:k]].mean()


MI = mutual_information(msa_fit)                           # deep alignment: all 2,000 fit sequences
MI_500 = mutual_information(msa_fit[:500])                  # a shallower alignment, like many real families
n_cand = len(np.triu_indices(L, 6)[0])
print(f"random ranking: precision = {long_range.sum() / 2 / n_cand:.3f}  ({long_range.sum() // 2} contacts among {n_cand} pairs)")
print("                 2,000 sequences   500 sequences")
print(f"MI        : precision@L/2 = {precision_at_k(MI):.2f}            {precision_at_k(MI_500):.2f}")
print(f"MI + APC  : precision@L/2 = {precision_at_k(apc_correct(MI)):.2f}            {precision_at_k(apc_correct(MI_500)):.2f}")

In [ ]:
# ── 3.3 What APC removes: column entropy inflates raw MI ──
def plot_top_pairs(ax, scores, title, k=L // 2):
    """Grey = true contacts; top-k predicted pairs (|i − j| ≥ 6) in the upper triangle: blue = right, red = wrong."""
    ax.imshow(contacts, cmap="Greys", alpha=0.35)
    iu = np.triu_indices(L, 6)
    top = np.argsort(-scores[iu])[:k]
    ii, jj = iu[0][top], iu[1][top]
    ok = contacts[ii, jj]
    ax.scatter(jj[ok], ii[ok], s=12, color=OKABE_ITO[0]); ax.scatter(jj[~ok], ii[~ok], s=12, marker="x", color=OKABE_ITO[1])
    ax.set_title(f"{title}\nprecision@L/2 = {ok.mean():.2f}"); ax.set_xlabel("j"); ax.set_ylabel("i")


fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
plot_top_pairs(axes[0], MI_500, "raw MI (500 sequences)")
plot_top_pairs(axes[1], apc_correct(MI_500), "MI + APC (500 sequences)")
f_500 = column_frequencies(msa_fit[:500])
ent_500 = -(f_500 * np.log2(np.where(f_500 > 0, f_500, 1))).sum(1)
axes[2].scatter(ent_500, MI_500.sum(1) / (L - 1), s=14, color=OKABE_ITO[2])
axes[2].set_xlabel("column entropy (bits)"); axes[2].set_ylabel("mean MI with all other columns")
axes[2].set_title(f"the background APC removes: ρ = {spearmanr(ent_500, MI_500.sum(1))[0]:.2f}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With all 2,000 sequences both versions are perfect: the family is deep and its couplings are
strong. With 500 sequences raw MI makes systematic mistakes (left panel): the right panel shows that variable
(high-entropy) columns have higher MI with *every* other column — partly a small-sample effect (more possible residue
pairs, more chance co-occurrences) and partly the coupling network itself. Those columns pull non-contacts into the top
list. APC subtracts each pair's expected background and the precision jumps (the two numbers in the right-hand column
of cell 3.2). APC is a crude correction — but cheap, and still used on top of almost every coevolution and
attention-based contact score (Module 24 used it on attention maps).

**The deeper problem: indirect correlation.** Suppose residue 1 touches residue 2 and residue 2 touches residue 3, but
1 and 3 are far apart. Columns 1 and 3 still co-vary — through 2 — so MI(1, 3) > 0 although there is no direct contact.
In a dense contact network such chains are everywhere. This is exactly the difference between correlation and partial
correlation in [Module 18](18_gaussian_graphical_models.ipynb): the covariance matrix shows every connected pair, the
*precision* (inverse covariance) matrix only the direct edges. Its categorical analogue is the inverse of the covariance
matrix of the one-hot encoded columns (one residue per column is dropped, because the 20 indicators of a column sum to 1
and their covariance matrix would not be invertible). Let us check on three residues, computing everything exactly
from the $20^3=8{,}000$ possible sequences.

In [ ]:
# ── 3.4 Indirect correlation on a chain 1 – 2 – 3, computed exactly ──
rng = np.random.default_rng(SEED + 3)
J12, J23 = rng.normal(0, 1.5, (Q, Q)), rng.normal(0, 1.5, (Q, Q))     # 1 touches 2, 2 touches 3; J13 = 0
logP = J12[:, :, None] + J23[None, :, :]                              # log P(a, b, c) + constant, 20 × 20 × 20
P = np.exp(logP - lse(logP))
pair = {(0, 1): P.sum(2), (1, 2): P.sum(0), (0, 2): P.sum(1)}          # exact pairwise marginals
single = [P.sum((1, 2)), P.sum((0, 2)), P.sum((0, 1))]
mi_exact = {k: (p * np.log(p / np.outer(p.sum(1), p.sum(0)))).sum() for k, p in pair.items()}
m = Q - 1                                                             # drop the last residue of each column
C3 = np.zeros((3 * m, 3 * m))
for a in range(3):
    for b in range(3):
        block = np.diag(single[a]) if a == b else (pair[(a, b)] if a < b else pair[(b, a)].T)
        C3[a * m:(a + 1) * m, b * m:(b + 1) * m] = block[:m, :m] - np.outer(single[a][:m], single[b][:m])
J_mf = -np.linalg.inv(C3)                                             # "precision matrix" of the one-hot columns
norm = lambda a, b: np.sqrt((J_mf[a * m:(a + 1) * m, b * m:(b + 1) * m] ** 2).sum())
for (a, b), lab in [((0, 1), "1–2 (contact)"), ((1, 2), "2–3 (contact)"), ((0, 2), "1–3 (NO contact)")]:
    print(f"pair {lab:17s}: MI = {mi_exact[(a, b)]:.3f} nats | size of inverse-covariance block = {norm(a, b):.2e}")

🔍 **What just happened?** MI(1, 3) is far from zero — about one sixth of MI(1, 2) — even though residues 1 and 3
do not interact: MI cannot tell direct from indirect. The inverse covariance block for (1, 3), by contrast, is zero up to
rounding ($\sim10^{-11}$), while the two real contacts have large blocks. For tree-shaped graphs such as this chain the
zero is exact (Loh & Wainwright 2013 proved when the graph structure appears in the inverse of such generalised
covariance matrices); for graphs with loops it holds approximately, which is the basis of **mean-field DCA** below.

> 🔗 **Builds on** [Module 18 · Gaussian graphical models](18_gaussian_graphical_models.ipynb), Section 5: "the zeros
> of the precision matrix, not of the covariance matrix, are the missing edges". DCA applies the same idea to
> categorical variables.

✏️ **Your turn.** In cell 3.4 change the standard deviation of the couplings from 1.5 to 0.5. How does MI(1, 3)
change relative to MI(1, 2)? (Indirect correlation is a product of two links, so it fades quickly when the links are weak.)

---
## 4 · Potts models and direct-coupling analysis

> 📍 **Where we are.** We want a model of the whole family in which *direct* pair interactions are explicit
> parameters. That is the Potts model — the model the toolkit used to generate our data. Fitting it to the alignment
> and reading off the couplings is **direct-coupling analysis**.

🧮 **The Potts model.**

$$P(x_1,\dots,x_L)=\frac1Z\exp\Big(\sum_{i=1}^{L}h_i(x_i)+\sum_{i<j}J_{ij}(x_i,x_j)\Big),\qquad
Z=\sum_{x\in\{1..20\}^L}\exp\Big(\sum_i h_i(x_i)+\sum_{i<j}J_{ij}(x_i,x_j)\Big).$$

*In words:* fields $h_i$ encode conservation, couplings $J_{ij}$ encode which residue pairs fit together, and $Z$ (the
**partition function**) normalises the distribution. It is the maximum-entropy distribution that reproduces the
observed single- and pair-column frequencies, and it is a Markov random field
([Module 18](18_gaussian_graphical_models.ipynb)) whose graph is the set of non-zero couplings. Two remarks:

* **The partition function is intractable.** $Z$ sums over $20^{68}\approx3\times10^{88}$ sequences — more than the
  number of atoms in the observable universe. The likelihood $\prod_nP(x^{(n)})$ contains $Z$, so it cannot be
  maximised directly. DCA uses two approximations: mean field and pseudo-likelihood.
* **Gauge freedom.** Adding a constant to all entries of row $a$ of $J_{ij}$ and subtracting it from $h_i(a)$ leaves
  $P(x)$ unchanged, so couplings are only defined up to such shifts. Before comparing couplings we put them in the
  **zero-sum gauge** (every row and column of each $20\times20$ block sums to zero) and measure the strength of pair
  $(i,j)$ by the Frobenius norm $\lVert J_{ij}\rVert=\sqrt{\sum_{a,b}J_{ij}(a,b)^2}$.

**Mean-field DCA** (Morcos *et al.* 2011) uses the result of Section 3.4: $J\approx-C^{-1}$, where $C$ is the covariance
matrix of the one-hot columns (with a large pseudocount, which acts as regularisation). One matrix inversion of size
$68\cdot19=1{,}292$.

In [ ]:
# ── 4.1 Mean-field DCA ──
def mean_field_dca(msa, pseudocount=0.5):
    """Couplings J = −C⁻¹ (C = connected correlations of one-hot columns, last residue dropped) — Morcos et al. 2011."""
    fi, fij = pair_frequencies(msa, pseudocount)
    C = (fij - fi[:, None, :, None] * fi[None, :, None, :])[:, :, :Q - 1, :Q - 1]
    C = C.transpose(0, 2, 1, 3).reshape(L * (Q - 1), L * (Q - 1))
    J = -np.linalg.inv(C).reshape(L, Q - 1, L, Q - 1).transpose(0, 2, 1, 3)
    return np.pad(J, ((0, 0), (0, 0), (0, 1), (0, 1)))                 # dropped residue: coupling 0 (a gauge choice)


def zero_sum(J):
    """Zero-sum gauge: remove row and column means of every 20 × 20 block (axes 2 and 3)."""
    return J - J.mean(2, keepdims=True) - J.mean(3, keepdims=True) + J.mean((2, 3), keepdims=True)


def coupling_scores(J):
    """Frobenius norm of every block in the zero-sum gauge, symmetrised, diagonal 0."""
    S = np.sqrt((zero_sum(J) ** 2).sum(axis=(2, 3)))
    S = (S + S.T) / 2
    np.fill_diagonal(S, 0)
    return S


t0 = time.time()
J_mf = mean_field_dca(msa_fit)
score_mf = apc_correct(coupling_scores(J_mf))
print(f"mean-field DCA in {time.time() - t0:.1f} s: precision@L/2 = {precision_at_k(score_mf):.2f}")

**Pseudo-likelihood DCA** (plmDCA; Ekeberg *et al.* 2013, building on Besag 1975). Instead of the joint likelihood,
maximise the product of the *conditional* probabilities of each residue given all the others — which, for a Potts
model, are simple softmaxes with no partition function (the same formula as the oracle of Module 24, Section 4):

$$\ell_{\text{PL}}(h,J)=\frac1N\sum_{n=1}^{N}\sum_{i=1}^{L}\log
\frac{\exp\big(h_i(x^{(n)}_i)+\sum_{j\ne i}J_{ij}(x^{(n)}_i,x^{(n)}_j)\big)}{\sum_{a}\exp\big(h_i(a)+\sum_{j\ne i}J_{ij}(a,x^{(n)}_j)\big)}
\;-\;\lambda\big(\lVert h\rVert^2+\lVert J\rVert^2\big).$$

*In words:* for every sequence and every position, ask how well the fields and couplings predict the observed residue
from the rest of the sequence — 68 small 20-class logistic regressions that share their coupling parameters — and add
an L2 (ridge, [Module 07](07_ridge_lasso_sparse_regression.ipynb)) penalty. Pseudo-likelihood is a *consistent*
estimator: with enough data it recovers the true parameters, even though it never computes $Z$. In code the 68
regressions are one matrix product: the one-hot alignment ($N\times1{,}360$) times a symmetric $1{,}360\times1{,}360$
coupling matrix.

In [ ]:
# ── 4.2 Pseudo-likelihood DCA in PyTorch (≈ 10–30 s) ──
def fit_plm_dca(msa, steps=100, lam=0.01, lr=0.05):
    """Maximise the L2-penalised pseudo-likelihood with Adam. Returns h (L × 20), J (L × L × 20 × 20) and the loss path."""
    N = len(msa)
    X = F.one_hot(torch.tensor(msa), Q).float().reshape(N, L * Q)
    y = torch.tensor(msa)
    h = torch.zeros(L, Q, requires_grad=True)
    W = torch.zeros(L * Q, L * Q, requires_grad=True)
    off_block = 1 - torch.kron(torch.eye(L), torch.ones(Q, Q))          # no coupling of a position with itself
    opt, path = torch.optim.Adam([h, W], lr=lr), []
    for step in range(steps):
        Jm = (W + W.T) / 2 * off_block                                    # symmetric: J_ij(a, b) = J_ji(b, a)
        logits = (X @ Jm).view(N, L, Q) + h                               # h_i(a) + Σ_j J_ij(a, x_j) for all n, i, a
        nll = F.cross_entropy(logits.reshape(-1, Q), y.reshape(-1), reduction="sum") / N
        loss = nll + lam * ((Jm ** 2).sum() + (h ** 2).sum())
        opt.zero_grad(); loss.backward(); opt.step()
        path.append(nll.item() / L)
    J = ((W + W.T) / 2 * off_block).detach().numpy().reshape(L, Q, L, Q).transpose(0, 2, 1, 3)
    return h.detach().numpy(), J, np.array(path)


torch.manual_seed(SEED)
t0 = time.time()
h_plm, J_plm, plm_path = fit_plm_dca(msa_fit)
score_plm = apc_correct(coupling_scores(J_plm))
print(f"plmDCA on {len(msa_fit)} sequences in {time.time() - t0:.0f} s; "
      f"pseudo-NLL per residue: step 1 {plm_path[0]:.3f} → step 100 {plm_path[-1]:.3f} nats")
print(f"plmDCA: precision@L/2 = {precision_at_k(score_plm):.2f}")

In [ ]:
# ── 4.3 🧪 Ground-truth check: contacts, couplings and held-out prediction ──
cp = np.array([(i, j) for i in range(L) for j in range(i + 1, L) if contacts[i, j]])   # the truly coupled pairs
true_blocks = zero_sum(J_true)[cp[:, 0], cp[:, 1]].ravel()
coupling_r = lambda J: pearsonr(zero_sum(J)[cp[:, 0], cp[:, 1]].ravel(), true_blocks)[0]


def conditional_log_probs(X, h, J):
    """log p(x_i = a | rest) under a Potts model with parameters (h, J), for every sequence and position."""
    logits = np.repeat(h[None], len(X), axis=0)
    for j in range(L):
        logits += J[:, j][:, :, X[:, j]].transpose(2, 0, 1)
    return logits - lse(logits, axis=-1, keepdims=True)


held = msa_held[:300]
r_, c_ = np.arange(len(held))[:, None], np.arange(L)[None, :]
ppl = lambda lp: np.exp(-lp[r_, c_, held].mean())
rows = [("MI", precision_at_k(MI), np.nan, np.nan),
        ("MI + APC", precision_at_k(apc_correct(MI)), np.nan, np.nan),
        ("independent-site", np.nan, np.nan, ppl(np.repeat(log_f[None], len(held), axis=0))),
        ("mean-field DCA + APC", precision_at_k(score_mf), coupling_r(J_mf), np.nan),
        ("plmDCA + APC", precision_at_k(score_plm), coupling_r(J_plm), ppl(conditional_log_probs(held, h_plm, J_plm))),
        ("true Potts model", precision_at_k(apc_correct(coupling_scores(J_true))), 1.0,
         ppl(conditional_log_probs(held, h_true, J_true)))]
table = pd.DataFrame(rows, columns=["method", "contact precision@L/2", "r(inferred, true couplings)",
                                    "held-out perplexity (one residue hidden)"])
print(table.round(3).to_string(index=False))

In [ ]:
# ── 4.4 plmDCA convergence, and inferred vs true couplings ──
fig, (a0, a1, a2) = plt.subplots(1, 3, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1, 1, 1.3]})
a0.plot(np.arange(1, len(plm_path) + 1), plm_path, color=OKABE_ITO[0])
a0.set_xlabel("Adam step"); a0.set_ylabel("pseudo-NLL per residue (nats)")
a0.set_title(f"plmDCA convergence\nchange over the last 20 steps: {abs(plm_path[-20] - plm_path[-1]):.1e}")
plot_top_pairs(a1, score_plm, "plmDCA + APC")
inferred = zero_sum(J_plm)[cp[:, 0], cp[:, 1]].ravel()
a2.scatter(true_blocks, inferred, s=2, alpha=0.3, color=OKABE_ITO[0])
lim, slope = np.abs(true_blocks).max(), np.polyfit(true_blocks, inferred, 1)[0]
a2.plot([-lim, lim], [-lim * slope, lim * slope], color="k", lw=0.8, ls="--", label=f"least-squares slope {slope:.2f}")
a2.legend(loc="upper left")
a2.set_xlabel("true coupling $J_{ij}(a,b)$ (zero-sum gauge)"); a2.set_ylabel("plmDCA coupling")
a2.set_title(f"all 400 entries of the {len(cp)} contact blocks\nPearson r = {coupling_r(J_plm):.2f}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Read the table row by row (and the left panel first).

* **Convergence.** The pseudo-NLL (the data term of the objective) falls fast, overshoots — the L2 penalty then pulls the
  couplings back — and is flat long before step 100 (the change over the last 20 steps is printed in the panel title), so
  what follows describes the penalised optimum, not an optimisation stopped too early.

* **Contacts.** With 2,000 sequences every method, even raw MI, puts only true contacts in its top 34 pairs (base rate
  0.019). For such a deep family contact precision cannot separate the methods (Section 4.5 makes the task harder).
* **Couplings.** Only DCA estimates the coupling *tables*, and only moderately well: their entries correlate with the
  true ones with $r\approx0.55$ for both variants, and plmDCA's are strongly shrunk (least-squares slope about 0.07 in
  the right panel). Each of the 400 entries of a table is supported by only a handful of the 2,000 sequences, so the
  L2 penalty dominates; larger penalties shrink the tables further and smaller ones over-fit (✏️ below). Contact
  ranking survives the shrinkage because it compares the overall size of whole tables.
* **Held-out prediction.** The fitted Potts model predicts a hidden residue only slightly better than the
  independent-site model (perplexity about 12.4 vs 13.0) and far worse than the true model (9.2). Knowing *which* pairs
  are coupled is much easier than knowing *how*: about 0.9 million coupling parameters are estimated from 136,000
  observed residues.

Now the hard regime. Real families range from millions of sequences to a handful, and coevolution needs depth. We
refit everything on subsets of the alignment.

In [ ]:
# ── 4.5 🧪 How deep must an alignment be? Precision vs number of sequences ──
depths = [50, 100, 200, 400, 800, 2000]
curve = {"raw MI": [], "MI + APC": [], "mean-field DCA": [], "plmDCA": []}
t0 = time.time()
for n in depths:
    sub = msa_fit[:n]
    mi_sub = mutual_information(sub)
    curve["raw MI"].append(precision_at_k(mi_sub))
    curve["MI + APC"].append(precision_at_k(apc_correct(mi_sub)))
    curve["mean-field DCA"].append(precision_at_k(apc_correct(coupling_scores(mean_field_dca(sub)))))
    curve["plmDCA"].append(precision_at_k(score_plm) if n == len(msa_fit) else
                           precision_at_k(apc_correct(coupling_scores(fit_plm_dca(sub, steps=100)[1]))))
print(f"refits done in {time.time() - t0:.0f} s")
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for (name, vals), c, mk in zip(curve.items(), [OKABE_ITO[3], OKABE_ITO[0], OKABE_ITO[2], OKABE_ITO[1]], "^osD"):
    ax.plot(depths, vals, marker=mk, color=c, label=name)
ax.axhline(long_range.sum() / 2 / n_cand, color="k", ls=":", lw=0.8, label="random")
ax.set_xscale("log"); ax.set_xlabel("number of sequences in the alignment"); ax.set_ylabel("precision@L/2")
ax.set_title("SIMULATED family: contact prediction needs deep alignments"); ax.legend()
plt.tight_layout(); plt.show()
print(pd.DataFrame(curve, index=depths).round(2).T.to_string())

🔍 **What just happened?** With 50–100 sequences no method gets more than about a fifth of its top pairs right; between
200 and 800 sequences precision climbs steeply; from 800 on the corrected methods are (almost) perfect. Two lessons.
First, **APC is essential**: raw MI needs roughly twice as many sequences as any corrected method for the same precision.
Second, in *this* simulation APC-corrected MI is nearly as good as DCA: plmDCA is best at 400 sequences (0.82 against
0.71 for MI + APC and 0.68 for mean-field DCA), and at 200 sequences the three are within a few hundredths. Our couplings
are random tables of moderate strength, so indirect correlations stay weak (Section 3.4: they are products of links); in
real protein families, global models such as DCA predicted contacts more accurately than MI-based scores (Morcos *et
al.* 2011). For real proteins the useful quantity is $N_{\text{eff}}$ per residue: here precision becomes high at
roughly 6–12 sequences per residue (400–800 for $L=68$), and real families likewise need several effective sequences
per residue (Kamisetty *et al.* 2013 analysed this question) — which is why coevolution worked for large families first,
and why metagenomic sequencing (more sequences) and deep learning (better use of few sequences) mattered so much.

⚠️ **Pitfall.** Precision@L/2 depends on the protein's length and contact density; compare methods on the same protein
and the same definition of "long-range". Our $|i-j|\ge6$ is lenient; CASP reports long-range contacts with $|i-j|\ge24$.

✏️ **Your turn.** Change `lam` in `fit_plm_dca` to 0.1 and to 0.001 for the 200-sequence refit. Which value gives the
best contacts? Is the best value for contacts also the best for the coupling correlation?

### 📄 Research connection — Coevolution and direct-coupling analysis

- **What the field does.** Mutual information between alignment columns was the first coevolution signal used for contact
    prediction; Dunn, Wahl & Gloor (2008) showed that the average product correction removes much of its entropy and
    phylogeny background. Weigt *et al.* (2009) introduced direct-coupling analysis to separate direct from indirect
    correlations; Morcos *et al.* (2011) made it fast with the mean-field approximation, and Marks *et al.* (2011) used
    the predicted contacts to fold proteins of unknown structure (EVfold). Pseudo-likelihood DCA (Ekeberg *et al.*
    2013) and GREMLIN (Kamisetty, Ovchinnikov & Baker 2013) improved accuracy, especially for smaller families.
- **What this notebook builds.** MI, APC, mean-field DCA (one matrix inversion) and pseudo-likelihood DCA (a PyTorch optimisation with an L2
    penalty) on a simulated family with known couplings, evaluated by contact precision@L/2, correlation with the true
    coupling tables, held-out conditional likelihood and alignment depth. Production tools (plmc, CCMpred, EVcouplings)
    add sequence reweighting, gap handling and careful regularisation.
- **Where the frontier is.** Since about 2017–2020, deep networks trained on structures (RaptorX, trRosetta, AlphaFold) have
    replaced DCA as the most accurate contact predictors, but they still consume the alignment and its coevolution
    signal. Current questions include coevolution between *different* proteins to predict interaction partners (which
    requires pairing sequences across families), correcting for phylogeny, families with very few sequences, and models
    beyond pairs.
- **Read next.** Morcos *et al.* 2011 (PNAS); Ekeberg *et al.* 2013 (Phys Rev E); Marks *et al.* 2011 (PLoS ONE).

---
## 5 · Zero-shot variant effects and epistasis

> 📍 **Where we are.** A fitted Potts model is a model of the whole sequence, so it can score any mutant. The idea of
> EVmutation (Hopf *et al.* 2017) is to use the change in Potts score as a prediction of the mutation's effect.

$$\Delta E(x^{\text{wt}}\!\to x)=\sum_i\big[h_i(x_i)-h_i(x^{\text{wt}}_i)\big]+\sum_{i<j}\big[J_{ij}(x_i,x_j)-J_{ij}(x^{\text{wt}}_i,x^{\text{wt}}_j)\big].$$

*In words:* the effect of a mutation is how much the statistical "fitness" of the sequence changes; for a single
mutation at $i$ only the field of $i$ and the couplings of $i$ to its partners change, and the coupling terms depend on
the wild-type residues at those partners. $\Delta E<0$ means the mutant is less like a typical family member. This is
exactly the quantity the toolkit's `deep_mutational_scan` uses as the *true* effect (with the true $h$ and $J$).

In [ ]:
# ── 5.1 🧪 Single mutants: fitted Potts ΔE vs the independent-site model vs the truth ──
def single_mutant_effects(h, J, wt):
    """ΔE of every substitution wt_i → a under a Potts model (L × 20 matrix)."""
    partner = J[np.arange(L)[:, None], np.arange(L)[None, :], :, wt[None, :]].sum(axis=1)   # Σ_j J_ij(a, wt_j)
    return (h - h[np.arange(L), wt][:, None]) + (partner - partner[np.arange(L), wt][:, None])


dE_plm = single_mutant_effects(h_plm, J_plm, wild_type)
dms["potts"] = dE_plm[pos, mut]
print("Spearman correlation with ...           measured fitness   true ΔE")
for name in ["independent", "potts"]:
    print(f"  {name + ' score':28s}            {rho(dms[name], dms.fitness):.3f}        {rho(dms[name], dms.delta_energy):.3f}")
print(f"  {'true ΔE (ceiling)':28s}            {rho(dms.delta_energy, dms.fitness):.3f}        1.000")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, name, label, c in zip(axes, ["independent", "potts"], ["independent-site model", "Potts model (plmDCA)"],
                              [OKABE_ITO[3], OKABE_ITO[0]]):
    ax.scatter(dms[name], dms.delta_energy, s=4, alpha=0.4, color=c)
    ax.set_xlabel(f"zero-shot score: {label}")
    ax.set_title(f"{label}: Spearman ρ with true ΔE = {rho(dms[name], dms.delta_energy):.2f}")
axes[0].set_ylabel("true ΔE (SIMULATED)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The Potts score orders the mutants better than the independent-site score, against both the
measured fitness and the true effects (numbers above): the couplings to the wild-type partners carry information that
conservation alone misses. It is still far from perfect — 2,000 sequences give noisy estimates for rare residues at
conserved positions, and the L2 penalty shrinks the couplings.

**Double mutants and epistasis.** Two mutations at positions in contact interact. The **epistasis** of a double mutant is
the part of its effect that is not the sum of the single effects:

$$\varepsilon_{ij}(a,b)=\Delta E_{ij}(a,b)-\Delta E_i(a)-\Delta E_j(b)
=J_{ij}(a,b)-J_{ij}(a,x^{\text{wt}}_j)-J_{ij}(x^{\text{wt}}_i,b)+J_{ij}(x^{\text{wt}}_i,x^{\text{wt}}_j).$$

*In words:* under a Potts model, epistasis between $i$ and $j$ comes only from their coupling table, evaluated at the four
combinations of wild-type and mutant residues. Any additive model (independent sites, or a sum of single-mutant
predictions) predicts $\varepsilon=0$ by construction. Compensatory mutations — the XOR of
[Module 22](22_neural_networks.ipynb), Section 2 — are the extreme case.

In [ ]:
# ── 5.2 🧪 Double mutants at long-range contacts: additive vs Potts vs truth ──
rng = np.random.default_rng(SEED + 5)
lr_pairs = np.array([(i, j) for i, j in cp if j - i >= 6])
dE_true1 = single_mutant_effects(h_true, J_true, wild_type)
rows = []
for _ in range(600):
    i, j = lr_pairs[rng.integers(len(lr_pairs))]
    a, b = rng.integers(Q), rng.integers(Q)
    if a == wild_type[i] or b == wild_type[j]:
        continue
    x = wild_type.copy(); x[i], x[j] = a, b
    true_double = potts_score(x, h_true, J_true)[0] - potts_score(wild_type, h_true, J_true)[0]
    eps_hat = J_plm[i, j, a, b] - J_plm[i, j, a, wild_type[j]] - J_plm[i, j, wild_type[i], b] + J_plm[i, j, wild_type[i], wild_type[j]]
    rows.append((true_double, true_double - dE_true1[i, a] - dE_true1[j, b],
                 log_f[i, a] - log_f[i, wild_type[i]] + log_f[j, b] - log_f[j, wild_type[j]],
                 dE_plm[i, a] + dE_plm[j, b], dE_plm[i, a] + dE_plm[j, b] + eps_hat, eps_hat))
dbl = pd.DataFrame(rows, columns=["true", "true_epistasis", "independent_additive", "potts_additive", "potts", "potts_epistasis"])
print(f"{len(dbl)} double mutants at long-range contacts")
for col in ["independent_additive", "potts_additive", "potts"]:
    print(f"  Spearman ρ({col:21s}, true double-mutant ΔE) = {rho(dbl[col], dbl.true):.3f}")
print(f"  Pearson r(Potts epistasis, true epistasis) = {pearsonr(dbl.potts_epistasis, dbl.true_epistasis)[0]:.3f}"
      f"   (additive models: 0 by construction)")
print(f"  spread of the true epistasis: sd = {dbl.true_epistasis.std():.2f} vs sd of the true double-mutant ΔE = {dbl.true.std():.2f}")

In [ ]:
# ── 5.3 Epistasis predicted by the fitted Potts model ──
fig, ax = plt.subplots(figsize=(5.2, 3.8))
ax.scatter(dbl.true_epistasis, dbl.potts_epistasis, s=8, alpha=0.5, color=OKABE_ITO[4])
ax.axhline(0, color=OKABE_ITO[3], lw=2, label="any additive model (ε = 0)")
ax.set_xlabel("true epistasis ε (SIMULATED)"); ax.set_ylabel("plmDCA epistasis")
ax.set_title(f"double mutants at contacts: r = {pearsonr(dbl.potts_epistasis, dbl.true_epistasis)[0]:.2f}")
ax.legend(); plt.tight_layout(); plt.show()

🔍 **What just happened?** For double mutants at contacting positions, the fitted couplings predict the *direction and
size* of epistasis — correlated with the truth — while additive models predict none. The total double-mutant ranking
improves less, because epistasis is only part of the effect: its spread is smaller than the spread of the total
$\Delta E$ (printed above), so the sum of two single effects already orders most double mutants. Epistasis matters most
where it changes sign (a mutation that is harmful alone but tolerated with its partner), which is exactly where
additive predictors fail.

✏️ **Your turn.** Restrict the double mutants to those whose two single mutants are both harmful
(`dE_true1[i, a] < -1` and `dE_true1[j, b] < -1`). Is the true epistasis mostly positive (the second hit hurts less than
expected) or negative? Explain the answer from the formula above. (Hint: the wild type was selected for favourable
couplings, so the loss of $J_{ij}(x^{\text{wt}}_i,x^{\text{wt}}_j)$ is counted twice in the additive prediction but only once
in reality.) Section 7 shows a second, assay-level source of apparent epistasis.

### 📄 Research connection — Variant-effect prediction from evolution

- **What the field does.** EVmutation (Hopf *et al.* 2017) showed that the Potts-model $\Delta E$ fitted to a family alignment predicts
    deep-mutational-scan measurements better than site-independent models, and captures epistasis. DeepSequence
    (Riesselman, Ingraham & Marks 2018) replaced the Potts model by a variational autoencoder over sequences
    ([Module 27](27_variational_inference_vae.ipynb)), capturing higher-order dependencies. EVE (Frazer *et al.* 2021)
    used such family models to classify human disease variants without training on clinical labels. AlphaMissense
    (Cheng *et al.* 2023) combines a protein language model, structural context and population frequency data, and
    released predictions for essentially all possible human missense substitutions.
- **What this notebook builds.** The independent-site score and the Potts $\Delta E$ from pseudo-likelihood DCA, evaluated against a
    simulated DMS (measured fitness) and against the true effects, plus a double-mutant analysis that isolates the
    epistasis term — the core logic of EVmutation.
- **Where the frontier is.** How to calibrate scores into clinically meaningful evidence (the ACMG/AMP framework originally treated
    computational predictions as supporting evidence only; ClinGen's calibration, Pejaver *et al.* 2022, allows calibrated
    predictors up to moderate or strong evidence at stringent thresholds); how to avoid
    circularity when predictors are tuned on the same clinical databases used to evaluate them; how a protein-level
    effect relates to organism-level disease; and how well predictions transfer across genes and populations.
- **Read next.** Hopf *et al.* 2017 (Nat Biotechnol); Frazer *et al.* 2021 (Nature); Cheng *et al.* 2023 (Science).

---
## 6 · A protein language model

> 📍 **Where we are.** [Module 24](24_attention_transformers.ipynb) built a transformer, pre-trained it by masked
> residue prediction and scored mutations by **masked marginals** (Meier *et al.* 2021). Here we train the same
> architecture on this family and compare it with the Potts model on the same tasks.

Module 24 verified that its from-scratch attention equals PyTorch's built-in `nn.MultiheadAttention`, so we can use the
built-in pre-LayerNorm transformer layer to keep the code compact; the architecture (64-dimensional embeddings, 4 heads,
2 layers, MLP width 128, learned positions) and the BERT masking are identical.

In [ ]:
# ── 6.1 The Module 24 encoder, compactly ──
MASK = 20


class FamilyLM(nn.Module):
    """Token + learned position embeddings → 2 pre-LayerNorm transformer layers → logits over 20 residues."""

    def __init__(self, n_pos, d=64, heads=4, layers=2, d_ff=128):
        super().__init__()
        self.tok, self.pos = nn.Embedding(21, d), nn.Embedding(n_pos, d)
        layer = nn.TransformerEncoderLayer(d, heads, d_ff, dropout=0.0, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, 20)

    def forward(self, tokens):
        return self.head(self.ln(self.encoder(self.tok(tokens) + self.pos.weight[: tokens.shape[1]])))


def bert_mask(x, rate=0.15):
    """15 % of positions chosen; 80 % → [MASK], 10 % → random residue, 10 % unchanged."""
    chosen, r = torch.rand(x.shape) < rate, torch.rand(x.shape)
    x_in = x.clone()
    x_in[chosen & (r < 0.8)] = MASK
    swap = chosen & (r >= 0.8) & (r < 0.9)
    x_in[swap] = torch.randint(0, 20, (int(swap.sum()),))
    return x_in, chosen

In [ ]:
# ── 6.2 Masked-LM training on the 2,000 fit sequences (≈ 15–40 s) ──
torch.manual_seed(SEED)
plm = FamilyLM(L)
X_fit = torch.tensor(msa_fit)
opt = torch.optim.AdamW(plm.parameters(), lr=3e-3, weight_decay=0.01)
STEPS = 400
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=STEPS, pct_start=0.1)
t0 = time.time()
for step in range(STEPS):
    batch = X_fit[torch.randint(0, len(X_fit), (64,))]
    x_in, chosen = bert_mask(batch)
    loss = F.cross_entropy(plm(x_in)[chosen], batch[chosen])
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
print(f"{sum(p.numel() for p in plm.parameters()):,} parameters, {STEPS} steps in {time.time() - t0:.0f} s, "
      f"last batch loss {loss.item():.3f}")

In [ ]:
# ── 6.3 🧪 Held-out perplexity and zero-shot variant scores: language model vs Potts ──
@torch.no_grad()
def lm_single_site_log_probs(model, X):
    """log p(x_i = a | rest) from the language model, masking one position at a time."""
    model.eval()
    X = torch.tensor(X)
    out = torch.zeros(len(X), L, 20)
    for i in range(L):
        x_in = X.clone(); x_in[:, i] = MASK
        out[:, i] = torch.log_softmax(model(x_in)[:, i], -1)
    return out.numpy()


held_lm = held[:120]
r2, c2 = np.arange(len(held_lm))[:, None], np.arange(L)[None, :]
ppl2 = lambda lp: np.exp(-lp[r2, c2, held_lm].mean())
lp_lm = lm_single_site_log_probs(plm, held_lm)
print("held-out perplexity (one residue hidden at a time):")
print(f"  independent-site {ppl2(np.repeat(log_f[None], len(held_lm), axis=0)):.2f} | language model {ppl2(lp_lm):.2f} | "
      f"plmDCA {ppl2(conditional_log_probs(held_lm, h_plm, J_plm)):.2f} | true Potts {ppl2(conditional_log_probs(held_lm, h_true, J_true)):.2f}")
masked_wt = lm_single_site_log_probs(plm, wild_type[None])[0]                        # L × 20 masked marginals
dms["language_model"] = masked_wt[pos, mut] - masked_wt[pos, wt_aa]
print("\nSpearman correlation with ...           measured fitness   true ΔE")
for name in ["independent", "language_model", "potts"]:
    print(f"  {name:28s}            {rho(dms[name], dms.fitness):.3f}        {rho(dms[name], dms.delta_energy):.3f}")
print(f"  Spearman ρ(language model, independent-site score) = {rho(dms.language_model, dms.independent):.3f}")

🔍 **What just happened?** An honest negative result. On this family the language model behaves almost exactly like the
independent-site model: its held-out perplexity is about the same (no better), its zero-shot scores correlate with the
DMS no better than conservation, and its scores are highly correlated with the independent-site scores. The Potts model
beats it on every measure. In Module 24 the *same architecture*, trained the same way, captured almost 60 % of the
context information and recovered long-range contacts with precision@L/2 ≈ 0.6. What is different?

The couplings. In Module 24 every coupling table was a mixture of three shared "chemistry" matrices, so a few attention
heads could represent all of them. The toolkit's `simulate_protein_family` draws an **independent random**
$20\times20$ table for each of the 128 contacts — there is no shared rule to learn. Let us measure this directly: stack
the 128 true coupling tables as vectors of 400 numbers and ask how many dimensions are needed to describe them (a PCA,
[Module 14](14_pca_dimensionality_reduction.ipynb)).

In [ ]:
# ── 6.4 Why: how many dimensions do the coupling tables span? ──
HYDROPATHY = np.array([1.8, 2.5, -3.5, -3.5, 2.8, -0.4, -3.2, 4.5, -3.9, 3.8,
                       1.9, -3.5, -1.6, -3.5, -4.5, -0.8, -0.7, 4.2, -0.9, -1.3])    # Kyte–Doolittle (Module 24)
CHARGE = np.array([0, 0, -1, -1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], float)
VOLUME = np.array([88.6, 108.5, 111.1, 138.4, 189.9, 60.1, 153.2, 166.7, 168.6, 166.7,
                   162.9, 114.1, 112.7, 143.8, 173.4, 89.0, 116.1, 140.0, 227.8, 193.6])   # residue volume, Å³ (approx.)
zs_ = lambda v: (v - v.mean()) / v.std()
chem = np.stack([np.outer(zs_(HYDROPATHY), zs_(HYDROPATHY)), -np.outer(CHARGE, CHARGE), -np.outer(zs_(VOLUME), zs_(VOLUME))])
rng = np.random.default_rng(SEED + 6)
chem_blocks = np.stack([np.tensordot(rng.dirichlet([0.5] * 3), chem, 1) + rng.normal(0, 0.1, (Q, Q)) for _ in cp])


def variance_explained(blocks):
    """Cumulative fraction of variance of the (zero-sum) coupling tables captured by the top principal components."""
    B = zero_sum(blocks[:, None]).reshape(len(blocks), -1)
    s = np.linalg.svd(B - B.mean(0), compute_uv=False) ** 2
    return np.cumsum(s) / s.sum()


ve_toolkit = variance_explained(J_true[cp[:, 0], cp[:, 1]])
ve_chem = variance_explained(chem_blocks)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(np.arange(1, len(cp) + 1), ve_toolkit, color=OKABE_ITO[1], label="this family (random tables)")
ax.plot(np.arange(1, len(cp) + 1), ve_chem, color=OKABE_ITO[0], label="Module 24-style chemistry tables")
ax.set_xscale("log"); ax.set_xlabel("number of principal components"); ax.set_ylabel("variance explained")
ax.set_title(f"the {len(cp)} true coupling tables: shared structure or not?"); ax.legend()
plt.tight_layout(); plt.show()
print(f"variance explained by 3 components: random tables {ve_toolkit[2]:.2f} | chemistry tables {ve_chem[2]:.2f}")

🔍 **What just happened?** The chemistry tables of Module 24 live almost entirely in a 3-dimensional space; the
toolkit's random tables spread their variance over dozens of dimensions — three components capture only about 6 %.
To predict a residue from its partners, a model must store, for every contact, how each partner residue shifts each of
the 20 probabilities. The Potts model has a dedicated table for every pair of positions (2,278 pairs × 400 entries ≈ 0.9 million coupling
parameters, most of them shrunk towards zero), so it can. Our 74,000-parameter transformer shares its weights across all pairs, so it can only
learn couplings that follow shared rules — and here there are none. With 400 training steps it learned the column
frequencies and stopped there.

**Family-specific vs universal models.** This toy exaggerates a real distinction:

| | Potts / DCA / EVmutation (family-specific) | Protein language model (universal, e.g. ESM) |
|---|---|---|
| Training data | one alignment of the family | hundreds of millions of unaligned sequences of all families |
| Parameters | ~ $L^2q^2$ per family, no sharing between pairs | shared across all positions, pairs and families |
| Needs | a deep alignment ($N_{\text{eff}}\gg L$) | no alignment at prediction time |
| Strength | exact pairwise model when data are plentiful | transfers regularities (chemistry, motifs, structure) across families; works for shallow families |
| Weakness | fails for shallow or orphan families | must be large; may memorise families seen in training |

Real residue–residue preferences *are* largely shared (hydrophobic packing, charge complementarity, steric fit), and a
large model trained across families can learn them once and reuse them everywhere — part of why protein language models
work. On the deep mutational scans of ProteinGym, family-specific alignment models and protein language models perform in a
similar range on average, each better on some proteins than on others (Notin *et al.* 2023), and hybrids that combine
them are common.

✏️ **Your turn.** Try to make the language model learn this family: train for 1,200 steps, or with `d=128`. Does the
held-out perplexity move below the independent-site value? Estimate how many parameters would be needed to store 128
unrelated $20\times20$ tables, and compare with the model's size.

### 📄 Research connection — Protein language models and benchmarks

- **What the field does.** Protein language models (ESM-1b, Rives *et al.* 2021; ESM-1v, Meier *et al.* 2021; ESM-2, Lin *et al.*
    2023) are trained by masked-residue prediction on UniRef and score variants zero-shot by masked marginals. Deep
    mutational scanning (Fowler & Fields 2014) supplies the experimental ground truth, and ProteinGym (Notin *et al.*
    2023) collects hundreds of DMS assays and clinical variant sets as a common benchmark, reporting rank correlations per
    assay.
- **What this notebook builds.** A family-specific masked language model with exactly the Module 24 architecture, compared with plmDCA on
    held-out conditional prediction and zero-shot DMS scoring, and a PCA of the true coupling tables that explains *why*
    a small weight-sharing model fails on a family whose couplings follow no shared rule.
- **Where the frontier is.** Whether bigger language models keep improving variant-effect prediction (results in 2023–2025 suggest
    gains saturate or depend on how well a family is represented in the training data); combining sequence, alignment
    and structure (e.g. MSA-based and structure-aware models); and evaluation that separates families seen during
    pre-training from new ones.
- **Read next.** Meier *et al.* 2021 (NeurIPS); Notin *et al.* 2023, ProteinGym (NeurIPS Datasets & Benchmarks);
    Lin *et al.* 2023 (Science).

---
## 7 · Global epistasis and the non-linearity of the assay

> 📍 **Where we are.** All our comparisons used Spearman correlation. Why ranks? Because a DMS does not measure
> $\Delta E$ directly: the readout (growth, fluorescence, binding) is a **non-linear** function of the underlying
> effect. The toolkit's DMS is $\text{fitness}=\sigma\big((\Delta E+3)/1.5\big)+\text{noise}$, a sigmoid like the
> "fraction folded" of a protein whose stability changes.

This produces **global epistasis** (Otwinowski, McCandlish & Plotkin 2018): even if mutations add up perfectly on the
$\Delta E$ scale, the measured fitness does not add — a mutation in an already crippled protein changes the readout
little (the floor), and a stabilising mutation in a very stable protein too (the ceiling). Pearson correlation, which
assumes a straight line, is penalised by the curvature; Spearman is not. The fix, when we want calibrated predictions,
is to **fit a monotone link** from predictor to measurement — here a 4-parameter sigmoid and a non-parametric isotonic
regression — on part of the data and evaluate on the rest.

In [ ]:
# ── 7.1 🧪 Spearman vs Pearson, and fitting a monotone link on held-out mutants ──
def sigmoid_link(s, lo, hi, mid, width):
    """A 4-parameter sigmoid from a predictor s to the measured fitness."""
    return lo + (hi - lo) / (1 + np.exp(-(s - mid) / width))


rng = np.random.default_rng(SEED + 7)
train_m = rng.random(len(dms)) < 0.5                                   # fit the link on half of the mutants
s, y = dms.potts.to_numpy(), dms.fitness.to_numpy()
p0 = [y.min(), y.max(), np.median(s), s.std()]
params, _ = curve_fit(sigmoid_link, s[train_m], y[train_m], p0=p0, maxfev=20000)
iso = IsotonicRegression(out_of_bounds="clip").fit(s[train_m], y[train_m])
te = ~train_m
print("test half of the mutants          Pearson r   Spearman ρ")
print(f"  raw Potts ΔE                     {pearsonr(s[te], y[te])[0]:.3f}       {rho(s[te], y[te]):.3f}")
print(f"  sigmoid link (4 parameters)      {pearsonr(sigmoid_link(s[te], *params), y[te])[0]:.3f}       "
      f"{rho(sigmoid_link(s[te], *params), y[te]):.3f}")
print(f"  isotonic regression              {pearsonr(iso.predict(s[te]), y[te])[0]:.3f}       {rho(iso.predict(s[te]), y[te]):.3f}")
print(f"  (true ΔE, raw)                   {pearsonr(dms.delta_energy[te], y[te])[0]:.3f}       {rho(dms.delta_energy[te], y[te]):.3f}")
print(f"fitted sigmoid link: floor {params[0]:.2f}, ceiling {params[1]:.2f}, midpoint {params[2]:.2f}, width {params[3]:.2f}")

🔍 **What just happened?** For single mutants the curve hardly matters here: on the test half, the Pearson correlation of
the raw Potts $\Delta E$ with the fitness is even slightly above its Spearman correlation, and neither fitted link improves
it on held-out mutants (a monotone map cannot change the Spearman correlation at all). Most single mutants of our wild
type sit on the steep, nearly linear part of the sigmoid (left panel below), and the predictor's own errors are much
larger than the curvature. The fitted floor and ceiling
(printed) are fairly close to the true 0 and 1; the midpoint and width live on the *predictor's* scale, which the L2 penalty has shrunk
(Section 4), so they need not match the true −3 and 1.5.

**Where global epistasis bites: double mutants.** Combine two mutations at positions that do **not** interact at all —
no coupling, so their effects add exactly on the $\Delta E$ scale. On the *fitness* scale they still do not add: two
mildly harmful mutations can push the protein over the edge of the sigmoid, and two strongly harmful ones cannot make the
fitness lower than the floor. Comparing measured double-mutant fitness with the sum of single-mutant fitness effects
therefore reports "epistasis" that is purely an artefact of the readout. The cure (Otwinowski *et al.* 2018) is to add
effects on a latent scale and then apply the non-linearity.

In [ ]:
# ── 7.2 🧪 Global epistasis: double mutants at positions that do NOT interact ──
def readout(dE):
    """The toolkit's assay curve without noise: σ((ΔE + 3) / 1.5)."""
    return 1 / (1 + np.exp(-(dE + 3) / 1.5))


rng = np.random.default_rng(SEED + 8)
uncoupled = np.array([(i, j) for i in range(L) for j in range(i + 1, L) if not np.abs(J_true[i, j]).any()])
rows = []
while len(rows) < 500:
    i, j = uncoupled[rng.integers(len(uncoupled))]
    a, b = rng.integers(Q), rng.integers(Q)
    if a == wild_type[i] or b == wild_type[j]:
        continue
    x = wild_type.copy(); x[i], x[j] = a, b
    rows.append((dE_true1[i, a], dE_true1[j, b], potts_score(x, h_true, J_true)[0] - potts_score(wild_type, h_true, J_true)[0],
                 dE_plm[i, a], dE_plm[j, b]))
dd = pd.DataFrame(rows, columns=["dE_a", "dE_b", "dE_ab", "pred_a", "pred_b"])
dd["fitness"] = readout(dd.dE_ab)                                               # true double-mutant fitness
dd["additive_fitness"] = readout(dd.dE_a) + readout(dd.dE_b) - readout(0.0)     # "no epistasis" on the fitness scale
dd["pred_fitness_scale"] = sigmoid_link(dd.pred_a, *params) + sigmoid_link(dd.pred_b, *params) - sigmoid_link(0.0, *params)
dd["pred_latent_scale"] = sigmoid_link(dd.pred_a + dd.pred_b, *params)          # add ΔE first, then apply the link
eps_fit = dd.fitness - dd.additive_fitness
print(f"{len(dd)} double mutants at uncoupled positions; max |ΔE_ab − ΔE_a − ΔE_b| = {np.abs(dd.dE_ab - dd.dE_a - dd.dE_b).max():.1e}")
print(f"apparent epistasis on the fitness scale: sd = {eps_fit.std():.3f}; "
      f"{np.mean(np.abs(eps_fit) > 0.1):.0%} of double mutants deviate from additivity by more than 0.1")
print(f"additive fitness prediction below 0 (impossible) for {np.mean(dd.additive_fitness < 0):.0%} of double mutants")
print(f"with the TRUE single effects: Pearson r(sum of fitness effects, true fitness) = "
      f"{pearsonr(dd.additive_fitness, dd.fitness)[0]:.3f}; adding ΔE and applying the readout is exact")
print("predicting double-mutant fitness from plmDCA single-mutant predictions (Pearson r with the truth):")
print(f"  add predicted fitness effects             : {pearsonr(dd.pred_fitness_scale, dd.fitness)[0]:.3f}")
print(f"  add predicted ΔE, then apply the link     : {pearsonr(dd.pred_latent_scale, dd.fitness)[0]:.3f}")

In [ ]:
# ── 7.3 The readout curve, and why fitness effects do not add ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
grid = np.linspace(dms.delta_energy.min(), dms.delta_energy.max(), 200)
axes[0].scatter(dms.delta_energy, dms.fitness, s=4, alpha=0.3, color="0.5", label="single mutants")
axes[0].plot(grid, readout(grid), color="k", lw=2, label="true readout")
axes[0].set_xlabel("true ΔE"); axes[0].set_ylabel("measured fitness (SIMULATED)")
axes[0].set_title("single mutants: a saturating readout"); axes[0].legend()
axes[1].scatter(dd.additive_fitness, dd.fitness, s=6, alpha=0.5, color=OKABE_ITO[4])
lims = [min(dd.additive_fitness.min(), 0), 1]
axes[1].plot(lims, lims, "k--", lw=0.8, label="no epistasis")
axes[1].set_xlabel("sum of single-mutant fitness effects"); axes[1].set_ylabel("true double-mutant fitness")
axes[1].set_title("uncoupled double mutants: 'epistasis'\nfrom the readout alone"); axes[1].legend()
axes[2].scatter(dd.pred_fitness_scale, dd.fitness, s=6, alpha=0.4, color=OKABE_ITO[3], label="add fitness effects")
axes[2].scatter(dd.pred_latent_scale, dd.fitness, s=6, alpha=0.4, color=OKABE_ITO[0], label="add ΔE, then link")
axes[2].plot(lims, lims, "k--", lw=0.8)
axes[2].set_xlabel("prediction from plmDCA single mutants"); axes[2].set_ylabel("true double-mutant fitness")
axes[2].set_title("modelling additivity on the right scale"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The two mutations of every double mutant here are at uncoupled positions — their $\Delta E$
add up exactly (the printed maximum deviation is zero up to rounding) — yet on the fitness scale many double mutants
deviate clearly from additivity (middle panel): when both singles are mildly harmful the double is *worse* than the sum
predicts, and when both are strongly harmful the "additive" prediction even goes below zero, while the true fitness
stops at the floor (almost a quarter of the "additive" predictions are below zero). An analysis on the fitness scale would
report widespread epistasis that has nothing to do with interacting residues. With the true single effects, adding
$\Delta E$ and applying the readout is exact, while summing fitness effects is worse ($r\approx0.93$ instead of 1). With the
plmDCA predictions the latent-scale route is better too, but only slightly (right panel; about 0.74 vs 0.73), because
most of the remaining error comes from the single-mutant predictions themselves. This is the logic of global-epistasis
models: one latent additive (or Potts) layer plus one monotone non-linearity, both fitted to data.

⚠️ **Pitfall.** Fit the link on training mutants only; fitting a flexible link (isotonic regression has many free
steps) on all mutants and reporting the Pearson correlation on the same mutants over-states performance — the selection
bias of [Module 23](23_model_selection.ipynb).

✏️ **Your turn.** Fit the same sigmoid link to the *independent-site* score. Does a non-linear link close the gap between
the independent-site and the Potts predictor? Why can no monotone link change the Spearman ranking of the methods?

---
## 8 · 🧭 From toy to real data

> 📍 **Where we are.** Our family was ideal: 3,000 independent, gap-free, perfectly aligned sequences from a known Potts
> model. Here is what changes on real proteins (reading only — no downloads).

**Data resources.** Sequences: [UniProt](https://www.uniprot.org) and its UniRef clusters; metagenomic collections add
many more. Families and seed alignments: [InterPro/Pfam](https://www.ebi.ac.uk/interpro/). Alignment building:
jackhmmer ([HMMER](http://hmmer.org)) and HHblits. Structures to check contacts: the [PDB](https://www.rcsb.org) and
the [AlphaFold DB](https://alphafold.ebi.ac.uk). Measured variant effects: [MaveDB](https://www.mavedb.org) (DMS data
sets) and [ProteinGym](https://proteingym.org) (curated benchmark). Clinical interpretations:
[ClinVar](https://www.ncbi.nlm.nih.gov/clinvar/).

**Production tools.** EVcouplings (alignment, plmDCA, contacts, EVmutation) and plmc; CCMpred and GREMLIN; ESM-2 and
ESM-1v (masked-marginal scores); EVE; AlphaMissense (pre-computed predictions for human missense variants). Computing
an alignment for one protein takes minutes to hours; plmDCA on a few thousand sequences of length 300 takes minutes on
a CPU; ESM-1v scores for one protein take seconds to minutes on a GPU.

**The first three things that go wrong.**

1. **The alignment.** Too few effective sequences (Section 4.5), poor coverage of the region of interest, fragments,
   misaligned segments and gap-rich columns all degrade every downstream model; the choice of search depth (E-value or
   bit-score threshold) is a real tuning parameter, and phylogeny creates couplings that are not contacts.
2. **The assay is not the model.** A DMS measures a particular selection (binding, stability, growth in a particular
   condition) through a non-linear, noisy readout; evolution "measures" long-term fitness across conditions. Correlations
   vary widely between assays, so benchmarks report Spearman per assay and average across assays rather than pooling.
3. **From score to clinic.** A pathogenicity score is not a diagnosis. The ACMG/AMP guidelines (Richards *et al.* 2015)
   originally counted computational evidence only as supporting evidence (criteria PP3/BP4); ClinGen's calibration study
   (Pejaver *et al.* 2022) showed that, at stringent and carefully calibrated thresholds, some predictors can contribute
   up to moderate or strong evidence. Either way the thresholds must be calibrated against variants of known effect; predictors trained or tuned on clinical databases must be evaluated on variants they have not seen,
   and performance can differ between genes and between populations.

✏️ **Your turn.** For a protein you care about, estimate from public resources how many effective sequences per residue
its family has, whether a DMS of it exists in MaveDB or ProteinGym, and whether its structure (PDB or AlphaFold DB)
would let you check predicted contacts. Which section of this notebook would fail first on that protein?

## 📌 Key takeaways

- A family alignment records **conservation** (column statistics) and **co-variation** (pair statistics); real
  alignments must be **reweighted** for redundancy ($w_n=1/m_n$, $N_{\text{eff}}=\sum w_n$), and phylogeny remains a confounder.
- The **independent-site model** (PSSM/profile) scores mutations zero-shot from conservation alone — a strong baseline
  (Spearman ≈ 0.6 with the simulated DMS) but blind to the context of each wild type.
- **Mutual information** finds co-varying pairs but is inflated by column entropy (fixed by **APC**) and cannot separate
  direct from **indirect** correlations; the inverse covariance of one-hot columns can (exactly for a chain) — the
  categorical version of Module 18's precision matrix.
- The **Potts model** is the maximum-entropy pairwise model of a family; its partition function is intractable, so DCA
  uses **mean-field** ($J\approx-C^{-1}$) or **pseudo-likelihood** (68 coupled softmax regressions, L2-penalised).
  Validated against truth: near-perfect contacts for deep alignments, coupling tables recovered with moderate accuracy,
  and a steep dependence on alignment depth.
- The Potts $\Delta E$ predicts single-mutant effects better than conservation and predicts **epistasis** at contacts,
  which additive models cannot.
- A small **language model** learns couplings only if they follow shared rules: on random per-contact tables it reduces
  to the independent-site model, while large universal PLMs exploit the shared chemistry of real proteins.
- DMS readouts are **non-linear** (global epistasis): compare predictors by rank (Spearman) and fit a monotone link on
  held-out data when calibrated predictions are needed.

---
## ✅ Checkpoint

1. Why is APC applied to MI (and to DCA scores and attention maps)?
   <details><summary>Answer</summary>Some columns score high with *every* other column (high entropy, small-sample noise, phylogeny). APC subtracts the expected background $\bar S_{i\cdot}\bar S_{\cdot j}/\bar S_{\cdot\cdot}$, leaving the pair-specific part.</details>
2. Why does DCA use pseudo-likelihood instead of the likelihood?
   <details><summary>Answer</summary>The likelihood contains the partition function $Z$, a sum over $20^L$ sequences ($\approx3\times10^{88}$ for $L=68$). The conditional distribution of one residue given the rest is a 20-term softmax with no $Z$, so the product of conditionals can be computed and maximised; it is a consistent estimator.</details>
3. Columns 1 and 2 have joint frequencies $f(A,A)=f(B,B)=0.5$ and $f(A,B)=f(B,A)=0$ (two residues, always matched). Compute their MI.
   <details><summary>Answer</summary>$f_1(A)=f_2(A)=0.5$. MI $=2\times0.5\log\frac{0.5}{0.25}=\log 2\approx0.693$ nats (1 bit): knowing one column determines the other.</details>
4. In what sense is mean-field DCA 'a Gaussian graphical model for categorical variables'?
   <details><summary>Answer</summary>It one-hot encodes the columns, computes their covariance matrix and inverts it, reading couplings from the precision matrix — exactly what a GGM does with continuous variables (Module 18). Zeros of the inverse covariance indicate (approximately) missing direct couplings; the mean-field approximation justifies $J\approx-C^{-1}$.</details>
5. An alignment contains 50 unrelated sequences and 100 identical copies of one more sequence. What is $N_{\text{eff}}$ with 80 % identity weighting?
   <details><summary>Answer</summary>Each unrelated sequence has weight 1; each of the 100 copies has weight 1/100. $N_{\text{eff}}=50+1=51$.</details>
6. What does $\Delta E<0$ mean for a mutation, and why does the Potts $\Delta E$ of the same substitution differ between two wild types?
   <details><summary>Answer</summary>The mutant has a lower Potts score — it is less typical of the family — predicting a harmful effect. The coupling terms $J_{ij}(a,x_j^{wt})$ depend on the residues at the contact partners, which differ between wild types.</details>
7. Why do DMS benchmarks report Spearman rather than Pearson correlation?
   <details><summary>Answer</summary>The assay readout is a non-linear (often sigmoidal), assay-specific function of the underlying effect. Spearman depends only on ranks, so it is unchanged by any monotone readout; Pearson penalises correct predictors for the curvature.</details>
8. Code reading: in `fit_plm_dca`, why is `W` symmetrised and multiplied by `off_block`?
   <details><summary>Answer</summary>Symmetrising enforces $J_{ij}(a,b)=J_{ji}(b,a)$ (one coupling per pair, as in the Potts model). `off_block` zeroes the $20\times20$ blocks on the diagonal, so a residue is never used to predict itself.</details>

## 🧠 Questions & discussion

1. Our simulated family has no phylogeny. Describe how a strongly clustered phylogeny (a few clades with many close relatives each) could create co-varying pairs that are not in contact. Which of the corrections in this notebook would help, and which would not?
2. DCA was validated here against true couplings. On a real protein the 'truth' for contacts is a structure. What can go wrong when a predicted contact is absent from the crystal structure (think of oligomers, alternative conformations, ligand-mediated contacts)?
3. A Potts model predicts that a patient's variant is strongly damaging, but the variant is common in a healthy population. List possible explanations, and discuss what this means for using evolutionary scores as clinical evidence.
4. The language model failed on this family but succeeds on real ones. What does that tell you about the assumptions built into an architecture, and about simulations designed to test methods?
5. Variant-effect predictors are increasingly used to prioritise variants for patients from all ancestries. How would you test whether a predictor performs equally well across genes, protein families and populations?

## 🚀 Bonus — going further

- **Generate sequences.** Use Gibbs sampling ([Module 21](21_bayesian_learning_mcmc.ipynb)) with the fitted plmDCA parameters to generate new sequences, and check whether their pair frequencies match the family's. This is the starting point of Potts-based protein design ([Module 39](39_generative_design.ipynb)).
- **From contacts to structure.** Take the top-$L$ plmDCA contacts and fold the helix bundle with distance restraints as in [Module 38](38_protein_structure.ipynb).
- **Phylogeny-induced false couplings.** Simulate a family with a tree: start from one sequence, mutate it along a random binary tree with a site-independent model (no couplings at all), and run MI and DCA. How many 'contacts' do they find? Does reweighting help?
- **Language model on a chemistry family.** Replace `simulate_protein_family` by the chemistry-coupled sampler of [Module 24](24_attention_transformers.ipynb) (Section 1.3) and rerun Sections 4–6. Does the language model now catch up with the Potts model?
- **Real data.** Install `evcouplings` or use ESM-2 via Hugging Face on Colab, pick a protein with a DMS in ProteinGym, and compare the independent-site, EVmutation and ESM-1v scores by Spearman correlation.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 37 · Protein sequences: from coevolution to protein language models
Biological question: is a never-seen missense variant damaging? Which residues touch?
Data (simulated): simulate_helix_bundle (68 residues, contacts) + simulate_protein_family (3,000 sequences,
                  Potts model with random coupling tables on the contacts) + deep_mutational_scan of the wild type.
                  Ground truth: contacts, h, J, ΔE of every mutant, the sigmoid readout.
Method: PSSM; MI + APC; mean-field DCA; pseudo-likelihood DCA (PyTorch); Potts ΔE; masked-LM transformer; link fitting
Key equation: P(x) ∝ exp(Σ h_i(x_i) + Σ J_ij(x_i, x_j));  ΔE = score(mutant) − score(wild type)
Key parameters: pseudocount λ, identity threshold for weights, L2 penalty, alignment depth, APC on/off
Ground-truth check result: precision@L/2 (MI / MI+APC / mfDCA / plmDCA) = … ; r(couplings) = … ;
                           Spearman vs DMS (independent / Potts / LM) = … ; epistasis r = …
What changed when I modified … (depth / λ / pseudocount / threshold): …
One failure mode: (e.g. MI fooled by entropy and indirect chains; LM ≈ independent-site on random couplings)
Connection to research: EVcouplings / EVmutation (Hopf 2017), ESM-1v (Meier 2021), ProteinGym (Notin 2023)
```

## 📚 References

- Durbin R., Eddy S., Krogh A., Mitchison G. (1998) *Biological Sequence Analysis*. Cambridge University Press (profiles, profile HMMs, pseudocounts, sequence weighting).
- Besag J. (1975) Statistical analysis of non-lattice data. *The Statistician* 24:179–195 (pseudo-likelihood).
- Dunn S. D., Wahl L. M., Gloor G. B. (2008) Mutual information without the influence of phylogeny or entropy dramatically improves residue contact prediction. *Bioinformatics* 24:333–340.
- Weigt M., White R. A., Szurmant H., Hoch J. A., Hwa T. (2009) Identification of direct residue contacts in protein–protein interaction by message passing. *PNAS* 106:67–72.
- Morcos F. *et al.* (2011) Direct-coupling analysis of residue coevolution captures native contacts across many protein families. *PNAS* 108:E1293–E1301.
- Marks D. S. *et al.* (2011) Protein 3D structure computed from evolutionary sequence variation. *PLoS ONE* 6:e28766.
- Ekeberg M., Lövkvist C., Lan Y., Weigt M., Aurell E. (2013) Improved contact prediction in proteins: using pseudolikelihoods to infer Potts models. *Phys Rev E* 87:012707.
- Kamisetty H., Ovchinnikov S., Baker D. (2013) Assessing the utility of coevolution-based residue–residue contact predictions in a sequence- and structure-rich era. *PNAS* 110:15674–15679.
- Loh P.-L., Wainwright M. J. (2013) Structure estimation for discrete graphical models: generalized covariance matrices and their inverses. *Ann Stat* 41:3022–3049.
- Hopf T. A. *et al.* (2017) Mutation effects predicted from sequence co-variation. *Nat Biotechnol* 35:128–135.
- Riesselman A. J., Ingraham J. B., Marks D. S. (2018) Deep generative models of genetic variation capture the effects of mutations. *Nat Methods* 15:816–822.
- Otwinowski J., McCandlish D. M., Plotkin J. B. (2018) Inferring the shape of global epistasis. *PNAS* 115:E7550–E7558.
- Fowler D. M., Fields S. (2014) Deep mutational scanning: a new style of protein science. *Nat Methods* 11:801–807.
- Frazer J. *et al.* (2021) Disease variant prediction with deep generative models of evolutionary data. *Nature* 599:91–95.
- Rives A. *et al.* (2021) Biological structure and function emerge from scaling unsupervised learning to 250 million protein sequences. *PNAS* 118:e2016239118.
- Meier J. *et al.* (2021) Language models enable zero-shot prediction of the effects of mutations on protein function. *NeurIPS*.
- Lin Z. *et al.* (2023) Evolutionary-scale prediction of atomic-level protein structure with a language model. *Science* 379:1123–1130.
- Cheng J. *et al.* (2023) Accurate proteome-wide missense variant effect prediction with AlphaMissense. *Science* 381:eadg7492.
- Notin P. *et al.* (2023) ProteinGym: large-scale benchmarks for protein fitness prediction and design. *NeurIPS Datasets & Benchmarks*.
- Pejaver V. *et al.* (2022) Calibration of computational tools for missense variant pathogenicity classification and ClinGen recommendations for PP3/BP4 criteria. *Am J Hum Genet* 109:2163–2177.
- Richards S. *et al.* (2015) Standards and guidelines for the interpretation of sequence variants: a joint consensus recommendation of the American College of Medical Genetics and Genomics and the Association for Molecular Pathology. *Genet Med* 17:405–424.

## ➡️ Next

We predicted which residues touch; the next question is how a list of contacts and distances becomes a 3-D
structure, how structures are compared, and how AlphaFold turns alignments into coordinates:
**[Module 38 · Protein structure: geometry, contacts and structure prediction](38_protein_structure.ipynb)**.